In [ ]:
"""
Temporal Epidemiological Analysis of PRR Disease
=================================================
Computes and plots cumulative epidemiological curves (incidence, AUDPC,
mortality) and their rates of change for three lots, then fits six
classical epidemiological models to incidence and AUDPC data.

Lots
----
  - Donmatias  : field data  (Excel)
  - El Retiro  : field data (Excel)
  - La Ceja    : field data (Excel)

Outputs (saved to OUTPUT_DIR)
------------------------------
  Fig_Epi_Cumulative.png / .pdf     — 3×3 cumulative curves + rate of change
  Fig_Models_Incidence.png / .pdf   — model fits for incidence
  Fig_Models_AUDPC.png / .pdf       — model fits for AUDPC
  Table_ModelMetrics_Incidence.png / .pdf
  Table_ModelMetrics_AUDPC.png / .pdf
  model_fit_metrics.csv             — AIC, RMSE, R² for all models and lots

Dependencies
------------
    pip install pandas numpy scipy matplotlib openpyxl

Usage
-----
    python temporal_epidemio_analysis.py
"""

import os
import warnings
import numpy  as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot   as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker   as mticker
warnings.filterwarnings("ignore")


# =============================================================================
# 0. USER CONFIGURATION  — edit only this section
# =============================================================================

DONMATIAS_XLSX = "Taller_epi.xlsx"
EL_RETIRO_XLSX = "Data_ElRetiro_1338.xlsx"   # Data not included due to confidentiality
LA_CEJA_XLSX   = "Data_LaCeja_1666.xlsx"     # Data not included due to confidentiality

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Time points present in the data (days)
TIME_POINTS = list(range(0, 1201, 60))

# Incidence definition: plants with severity >= this threshold
INCIDENCE_THRESHOLD = 1

# Mortality definition: plants with severity >= this threshold
MORTALITY_THRESHOLD = 5

# Figure resolution
DPI = 400


# =============================================================================
# 1. DATA LOADING
# =============================================================================

def _normalize_cols(df):
    """Strip spaces from column names and normalise 'D 60' → 'D60'."""
    df.columns = [c.strip().replace(" ", "") for c in df.columns]
    return df


def lat_lon_to_metres(df, lat_col="Latitude", lon_col="Longitude"):
    """
    Convert geographic coordinates to local metric coordinates centred on
    the lot centroid.  Adds columns 'Northing' and 'Easting' (metres).
    """
    lat0 = df[lat_col].mean()
    lon0 = df[lon_col].mean()
    cos_lat = np.cos(np.radians(lat0))
    df = df.copy()
    df["Northing"] = (df[lat_col] - lat0) * 111_320
    df["Easting"]  = (df[lon_col] - lon0) * 111_320 * cos_lat
    return df


def load_lot(xlsx_path, label, has_utm=True):
    """
    Load a lot from an Excel file.
    has_utm=True  → expects Northing/Easting columns already in metres.
    has_utm=False → expects Latitude/Longitude and converts to metres.
    Returns a cleaned DataFrame.
    """
    df = pd.read_excel(xlsx_path)
    df = _normalize_cols(df)
    if not has_utm:
        df = lat_lon_to_metres(df)
    print(f"  [{label}] {len(df)} plants loaded from {xlsx_path}")
    return df


def get_disease_cols(df, time_points=TIME_POINTS):
    """Return sorted list of disease column names present in df."""
    return [f"D{t}" for t in time_points if f"D{t}" in df.columns]


# =============================================================================
# 2. EPIDEMIOLOGICAL CURVE COMPUTATION
# =============================================================================

def compute_curves(df, label, time_points=TIME_POINTS):
    """
    Compute per-time-point epidemiological metrics from the severity matrix.

    Returns a dict with keys:
        label, months, time_pts,
        inc, audpc, mort,         ← cumulative values
        rate_inc, rate_audpc, rate_mort  ← delta per 60-day interval
    """
    dcols = get_disease_cols(df, time_points)
    n     = len(df)
    sev   = df[dcols].values.astype(float)          # shape (n_plants, n_timepoints)
    months = [t / 30 for t in time_points]

    # ── Incidence (%) ─────────────────────────────────────────────────────────
    inc = (sev >= INCIDENCE_THRESHOLD).mean(axis=0) * 100

    # ── Cumulative AUDPC (trapezoidal, plant-averaged) ─────────────────────────
    audpc_t = np.zeros((n, len(time_points)))
    for i in range(1, len(time_points)):
        dt            = time_points[i] - time_points[i - 1]
        audpc_t[:, i] = audpc_t[:, i - 1] + 0.5 * (sev[:, i] + sev[:, i - 1]) * dt
    mean_audpc = audpc_t.mean(axis=0)

    # ── Mortality (%) ─────────────────────────────────────────────────────────
    mort = (sev >= MORTALITY_THRESHOLD).mean(axis=0) * 100

    # ── Rates of change (delta per period) ────────────────────────────────────
    rate_inc   = np.diff(inc,        prepend=inc[0])
    rate_audpc = np.diff(mean_audpc, prepend=mean_audpc[0])
    rate_mort  = np.diff(mort,       prepend=mort[0])

    print(f"  [{label}]  "
          f"Incidence(D1200)={inc[-1]:.1f}%  "
          f"AUDPC(D1200)={mean_audpc[-1]:.1f}  "
          f"Mortality(D1200)={mort[-1]:.1f}%")

    return dict(label=label, months=months, time_pts=time_points,
                inc=inc, audpc=mean_audpc, mort=mort,
                rate_inc=rate_inc, rate_audpc=rate_audpc, rate_mort=rate_mort)


# =============================================================================
# 3. EPIDEMIOLOGICAL MODELS
# =============================================================================
#
# All models operate on data normalised to [0, 1] to keep parameter scales
# comparable.  After fitting, predictions are back-transformed to original
# units (multiply by max_val).
#
# Model signatures:  f(t, *params) → y    (t in months, y in [0,1])
# ─────────────────────────────────────────────────────────────────────────────

def monomolecular(t, K, r):
    """Monomolecular (von Bertalanffy): K*(1 - exp(-r*t))"""
    return K * (1 - np.exp(-r * t))

def exponential_model(t, y0, r):
    """Exponential growth: y0*exp(r*t)"""
    return y0 * np.exp(r * t)

def logistic(t, K, r, t0):
    """Logistic (sigmoid): K / (1 + exp(-r*(t-t0)))"""
    return K / (1 + np.exp(-r * (t - t0)))

def gompertz(t, K, r, t0):
    """Gompertz: K*exp(-exp(-r*(t-t0)))"""
    return K * np.exp(-np.exp(-r * (t - t0)))

def weibull(t, K, lam, k):
    """Weibull CDF: K*(1 - exp(-(t/lam)^k))"""
    t_pos = np.maximum(t, 1e-9)
    return K * (1 - np.exp(-(t_pos / lam) ** k))

def log_logistic(t, K, alpha, beta):
    """Log-logistic: K / (1 + (t/alpha)^(-beta))"""
    t_pos = np.maximum(t, 1e-9)
    return K / (1 + (t_pos / alpha) ** (-beta))


# Model registry: {name: (function, lower_bounds, upper_bounds, p0)}
MODELS = {
    "Monomolecular": (monomolecular,   [0,   0  ], [1.5, 2  ], [0.80, 0.10      ]),
    "Exponential":   (exponential_model,[0,  0  ], [0.5, 1  ], [0.01, 0.05      ]),
    "Logistic":      (logistic,         [0,  0,0 ], [1.5,2,50], [0.80, 0.30, 20.0]),
    "Gompertz":      (gompertz,         [0,  0,0 ], [1.5,2,50], [0.80, 0.20, 15.0]),
    "Weibull":       (weibull,          [0,0.1,0.1],[1.5,100,10],[0.80,15.0, 1.5 ]),
    "Log-logistic":  (log_logistic,     [0,0.1,0.1],[1.5,100,20],[0.80,15.0, 2.0 ]),
}

# Visual style per model
MOD_CLR = {
    "Monomolecular": "#555555",
    "Exponential":   "#E67E22",
    "Logistic":      "#1B4F8A",
    "Gompertz":      "#8E44AD",
    "Weibull":       "#C0392B",
    "Log-logistic":  "#1E8449",
}
MOD_LS = {
    "Monomolecular": "-",
    "Exponential":   "--",
    "Logistic":      "-",
    "Gompertz":      "--",
    "Weibull":       "-.",
    "Log-logistic":  ":",
}


def _metrics(y_obs, y_pred, n_params):
    """Compute R², RMSE and AIC for a fitted model."""
    n      = len(y_obs)
    ss_res = np.sum((y_obs - y_pred) ** 2)
    ss_tot = np.sum((y_obs - y_obs.mean()) ** 2)
    r2     = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    rmse   = np.sqrt(ss_res / n)
    aic    = n * np.log(ss_res / n + 1e-12) + 2 * n_params
    return r2, rmse, aic


def fit_models(t_months, y_obs_raw):
    """
    Fit all six epidemiological models to the observed data.

    Parameters
    ----------
    t_months  : 1-D array of time in months (excluding t=0)
    y_obs_raw : 1-D array of observed values in original units

    Returns
    -------
    list of dicts, one per model:
        {Model, R², RMSE, AIC, params, y_pred}
    """
    t      = np.asarray(t_months, dtype=float)
    max_v  = y_obs_raw.max()
    y_norm = np.clip(y_obs_raw / max_v, 0, 1)
    results = []

    for name, (func, lb, ub, p0) in MODELS.items():
        try:
            popt, _ = curve_fit(func, t, y_norm,
                                p0=p0, bounds=(lb, ub), maxfev=8000)
            y_pred_norm = func(t, *popt)
            y_pred      = y_pred_norm * max_v
            r2, rmse, aic = _metrics(y_obs_raw, y_pred, len(popt))
            results.append(dict(Model=name, R2=round(r2, 4),
                                RMSE=round(rmse, 4), AIC=round(aic, 2),
                                params=popt, y_pred=y_pred))
        except Exception:
            results.append(dict(Model=name, R2=np.nan, RMSE=np.nan,
                                AIC=np.nan, params=None, y_pred=None))

    return results


def fit_all_lots(curves_list):
    """
    Run model fitting for incidence and AUDPC across all lots.

    Returns
    -------
    fit_results : dict  {label: {variable: [model_dicts]}}
    metrics_rows: list of dicts  (for the comparison table)
    """
    fit_results  = {}
    metrics_rows = []

    for c in curves_list:
        label = c["label"]
        fit_results[label] = {}
        t_fit = np.array(c["months"][1:])   # exclude t=0

        for var_key, y_full, var_label in [
            ("inc",   c["inc"],   "Incidence (%)"),
            ("audpc", c["audpc"], "AUDPC"),
        ]:
            y_obs = y_full[1:]
            res   = fit_models(t_fit, y_obs)
            fit_results[label][var_label] = res

            print(f"  [{label}] {var_label}:")
            for r in sorted(res, key=lambda x: x["AIC"] if not np.isnan(x["AIC"]) else 9999):
                print(f"    {r['Model']:<15}  R²={r['R2']:.4f}  "
                      f"RMSE={r['RMSE']:.4f}  AIC={r['AIC']:.2f}")

            for r in res:
                metrics_rows.append(dict(
                    Lot=label, Variable=var_label, Model=r["Model"],
                    **{"R²": r["R2"], "RMSE": r["RMSE"], "AIC": r["AIC"]}
                ))

    return fit_results, metrics_rows


# =============================================================================
# 4. GLOBAL PLOT STYLE
# =============================================================================

plt.rcParams.update({
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":         10,
    "axes.linewidth":    0.8,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size":  3.5,
    "ytick.major.size":  3.5,
    "xtick.direction":   "out",
    "ytick.direction":   "out",
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

LOT_CLR = {
    "Donmatias": "#1B4F8A",
    "El Retiro": "#C0392B",
    "La Ceja":   "#1E8449",
}
LOT_MK = {"Donmatias": "o", "El Retiro": "s", "La Ceja": "^"}

BLACK      = "#000000"
LGRAY      = "#CCCCCC"
ALPHA_FILL = 0.18
FS_TITLE   = 12
FS_LABEL   = 9.5
FS_TICK    = 8.5
FS_SMALL   = 8.0


def _save(fig, basename):
    for ext in ("png", "pdf"):
        path = os.path.join(OUTPUT_DIR, f"{basename}.{ext}")
        fig.savefig(path, dpi=DPI if ext == "png" else None,
                    bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {basename}.png / .pdf")


# =============================================================================
# 5. FIGURE — CUMULATIVE CURVES + RATE OF CHANGE
# =============================================================================

def figure_cumulative(curves_list):
    """
    3 rows (Incidence / AUDPC / Mortality) × 3 cols (lots).
    Left axis  = cumulative curve.
    Right axis = rate of change (dashed line + filled area).
    Municipality name shown only in top row.
    All text in black.  No panel letters.  No row labels.
    """
    rows_cfg = [
        ("inc",   "rate_inc",   "Cumulative incidence (%)", "Rate (% per 60 d)"),
        ("audpc", "rate_audpc", "Cumulative AUDPC",         "AUDPC rate (per 60 d)"),
        ("mort",  "rate_mort",  "Cumulative mortality (%)", "Rate (% per 60 d)"),
    ]

    months_arr = np.array(curves_list[0]["months"])

    fig = plt.figure(figsize=(18, 13), facecolor="white", dpi=DPI)
    gs  = gridspec.GridSpec(3, 3, figure=fig,
                             hspace=0.28, wspace=0.36,
                             left=0.07, right=0.97,
                             top=0.93, bottom=0.07)

    for row, (cum_key, rate_key, ylabel_cum, ylabel_rate) in enumerate(rows_cfg):
        for col, c in enumerate(curves_list):
            clr = LOT_CLR[c["label"]]
            mk  = LOT_MK[c["label"]]
            ax  = fig.add_subplot(gs[row, col])
            ax2 = ax.twinx()

            cum_v  = c[cum_key]
            rate_v = c[rate_key]

            # Rate of change — right axis
            ax2.plot(months_arr, rate_v, color=clr, lw=1.2,
                     ls="--", alpha=0.80, zorder=2)
            ax2.fill_between(months_arr, 0, rate_v,
                             color=clr, alpha=ALPHA_FILL, zorder=1)
            ax2.set_ylabel(ylabel_rate, fontsize=FS_LABEL, color=BLACK,
                           labelpad=3, rotation=270, va="bottom")
            ax2.tick_params(axis="y", labelcolor=BLACK, labelsize=FS_TICK)
            ax2.yaxis.set_major_locator(mticker.MaxNLocator(4))
            ax2.spines["right"].set_color(LGRAY)
            ax2.spines["right"].set_linewidth(0.7)
            ax2.spines["left"].set_visible(False)

            # Cumulative — left axis
            ax.plot(months_arr, cum_v, color=clr, lw=2.0, zorder=4)
            ax.scatter(months_arr[::2], cum_v[::2],
                       color=clr, marker=mk, s=30, zorder=5, linewidths=0)
            if col == 0:
                ax.set_ylabel(ylabel_cum, fontsize=FS_LABEL,
                              color=BLACK, labelpad=3)
            ax.tick_params(axis="y", labelcolor=BLACK, labelsize=FS_TICK)
            ax.yaxis.set_major_locator(mticker.MaxNLocator(5))
            ax.set_ylim(bottom=0)
            ax.set_axisbelow(True)
            ax.yaxis.grid(True, color=LGRAY, lw=0.4, ls=":")
            ax.set_facecolor("#FAFAFA")
            ax.set_xlabel("Time (months)", fontsize=FS_LABEL,
                          labelpad=3, color=BLACK)
            ax.set_xlim(0, months_arr.max())
            ax.xaxis.set_major_locator(mticker.MultipleLocator(6))
            ax.tick_params(axis="x", labelsize=FS_TICK, colors=BLACK)
            ax.spines["left"].set_color(LGRAY);   ax.spines["left"].set_linewidth(0.7)
            ax.spines["bottom"].set_color(LGRAY); ax.spines["bottom"].set_linewidth(0.7)

            # Municipality name — top row only
            if row == 0:
                ax.set_title(c["label"], fontsize=FS_TITLE,
                             fontweight="bold", color=BLACK, pad=6)

    fig.suptitle(
        "Cumulative epidemiological curves and rate of change\n"
        "Incidence (%), AUDPC and Mortality (%)",
        fontsize=12, fontweight="bold", y=0.995, color=BLACK,
    )
    _save(fig, "Fig_Epi_Cumulative")


# =============================================================================
# 6. FIGURES — MODEL FITS
# =============================================================================

def figure_model_fits(curves_list, fit_results, var_label, fig_tag):
    """
    1 row × 3 cols: observed data + six fitted model curves for one variable.
    """
    months_arr = np.array(curves_list[0]["months"])
    t_fit      = months_arr[1:]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                             facecolor="white", dpi=DPI)
    fig.subplots_adjust(hspace=0.38, wspace=0.32,
                        left=0.07, right=0.97, top=0.88, bottom=0.12)

    for col, c in enumerate(curves_list):
        ax    = axes[col]
        clr   = LOT_CLR[c["label"]]
        mk    = LOT_MK[c["label"]]
        y_obs = c["inc"][1:] if "Incidence" in var_label else c["audpc"][1:]

        # Observed data
        ax.scatter(t_fit, y_obs, color=clr, s=32, zorder=5,
                   marker=mk, edgecolors="white", linewidths=0.4,
                   label="Observed")

        # Fitted models
        res = fit_results[c["label"]][var_label]
        for r in res:
            if r["y_pred"] is None:
                continue
            ax.plot(t_fit, r["y_pred"],
                    color=MOD_CLR[r["Model"]],
                    lw=1.4, ls=MOD_LS[r["Model"]], zorder=3,
                    label=f"{r['Model']} (R²={r['R2']:.3f})")

        ax.set_title(c["label"], fontsize=FS_TITLE,
                     fontweight="bold", color=clr, pad=4)
        ax.set_xlabel("Time (months)", fontsize=FS_LABEL, labelpad=2)
        if col == 0:
            ax.set_ylabel(var_label, fontsize=FS_LABEL, labelpad=2)
        ax.tick_params(labelsize=FS_TICK)
        ax.set_xlim(0, months_arr.max())
        ax.set_ylim(bottom=0)
        ax.set_facecolor("#FAFAFA")
        ax.xaxis.set_major_locator(mticker.MultipleLocator(6))
        ax.yaxis.grid(True, color=LGRAY, lw=0.4, ls=":")
        ax.legend(fontsize=FS_SMALL, loc="upper left", framealpha=0.88,
                  edgecolor=LGRAY, fancybox=False, borderpad=0.5,
                  handlelength=1.4)

    fig.suptitle(f"Epidemiological model fits — {var_label}",
                 fontsize=12, fontweight="bold", y=0.995, color=BLACK)
    _save(fig, f"Fig_Models_{fig_tag}")


# =============================================================================
# 7. TABLE — MODEL METRICS
# =============================================================================

PERIOD_ROW_COLORS = {
    "Donmatias": "#D6E4F0",
    "El Retiro": "#FADBD8",
    "La Ceja":   "#D5F5E3",
}
BEST_ROW_COLORS = {
    "Donmatias": "#AED6F1",
    "El Retiro": "#F1948A",
    "La Ceja":   "#ABEBC6",
}


def figure_metrics_table(metrics_rows, var_label, fig_tag):
    """
    Render model comparison table (R², RMSE, AIC) for one variable.
    Best-fit model per lot (lowest AIC) is highlighted.
    """
    df = pd.DataFrame(metrics_rows)
    df = df[df["Variable"] == var_label].copy()
    df = df.sort_values(["Lot", "AIC"])

    # Identify best model per lot
    best_per_lot = (df.groupby("Lot")["AIC"].idxmin()
                      .map(lambda i: df.loc[i, "Model"])
                      .to_dict())

    cols = ["Lot", "Model", "R²", "RMSE", "AIC"]
    data = df[cols].round({"R²": 4, "RMSE": 4, "AIC": 2}).values.tolist()

    fig, ax = plt.subplots(figsize=(14, 7), facecolor="white")
    ax.axis("off")
    tbl = ax.table(cellText=data, colLabels=cols,
                   loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.0, 1.9)

    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor("#BBBBBB")
        cell.set_linewidth(0.5)
        if r == 0:
            cell.set_facecolor("#1F3864")
            cell.set_text_props(color="white", fontweight="bold", fontsize=10)
        else:
            row_d = data[r - 1]
            lot   = row_d[0]; model = row_d[1]
            if best_per_lot.get(lot) == model:
                cell.set_facecolor(BEST_ROW_COLORS.get(lot, "#DDDDDD"))
                cell.set_text_props(fontweight="bold")
            else:
                cell.set_facecolor(PERIOD_ROW_COLORS.get(lot, "white"))

    # Replace raw 'Best' marker with readable text
    for r in range(1, len(data) + 1):
        row_d = data[r - 1]
        if best_per_lot.get(row_d[0]) == row_d[1]:
            pass   # already highlighted via color

    ax.set_title(
        f"Model fit metrics — {var_label}\n"
        "(R², RMSE, AIC;  highlighted rows = best model per lot by AIC)",
        fontsize=11, fontweight="bold", pad=14, color="black",
    )
    plt.tight_layout()
    _save(fig, f"Table_ModelMetrics_{fig_tag}")


# =============================================================================
# 8. MAIN PIPELINE
# =============================================================================

if __name__ == "__main__":

    # ── Load lots ─────────────────────────────────────────────────────────────
    lots_to_load = [
        (DONMATIAS_XLSX, "Donmatias", True),   # has_utm=True (Northing/Easting)
        (EL_RETIRO_XLSX, "El Retiro", False),  # has_utm=False (Lat/Lon)
        (LA_CEJA_XLSX,   "La Ceja",   False),
    ]

    curves_list = []
    for xlsx, label, has_utm in lots_to_load:
        if xlsx is None or not os.path.exists(xlsx):
            print(f"  [{label}] file not found — skipped.")
            continue
        df = load_lot(xlsx, label, has_utm=has_utm)
        curves_list.append(compute_curves(df, label))

    if not curves_list:
        raise RuntimeError("No lot data found. Check file paths in Section 0.")

    # ── Figure 1: Cumulative curves + rate of change ──────────────────────────
    print("\n── Generating cumulative curves figure ──")
    figure_cumulative(curves_list)

    # ── Model fitting ─────────────────────────────────────────────────────────
    print("\n── Fitting epidemiological models ──")
    fit_results, metrics_rows = fit_all_lots(curves_list)

    # Save raw metrics CSV
    csv_path = os.path.join(OUTPUT_DIR, "model_fit_metrics.csv")
    pd.DataFrame(metrics_rows).to_csv(csv_path, index=False)
    print(f"  Metrics saved: {csv_path}")

    # ── Figure 2: Model fits — Incidence ──────────────────────────────────────
    print("\n── Generating model fit figures ──")
    figure_model_fits(curves_list, fit_results,
                      var_label="Incidence (%)", fig_tag="Incidence")
    figure_model_fits(curves_list, fit_results,
                      var_label="AUDPC",         fig_tag="AUDPC")

    # ── Tables: model metrics ─────────────────────────────────────────────────
    print("\n── Generating model metrics tables ──")
    figure_metrics_table(metrics_rows, "Incidence (%)", "Incidence")
    figure_metrics_table(metrics_rows, "AUDPC",         "AUDPC")

    print("\nAll analyses complete.")